In [ ]:
# !pip install folium pandas openpyxl

import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster


# Load and Clean files

In [ ]:
FILEPATH = "Database_Summary.xlsx"

df = pd.read_excel(FILEPATH, engine="openpyxl")

df["StartLat"] = pd.to_numeric(df["StartLat"], errors="coerce")
df["StartLon"] = pd.to_numeric(df["StartLon"], errors="coerce")

# remove missing and obviously invalid
dfp = df.dropna(subset=["StartLat", "StartLon"]).copy()
dfp = dfp[
    dfp["StartLat"].between(-90, 90) &
    dfp["StartLon"].between(-180, 180)
].copy()

# Optional: remove "0,0" points (common placeholder)
dfp = dfp[~((dfp["StartLat"].abs() < 1e-6) & (dfp["StartLon"].abs() < 1e-6))].copy()

dfp.shape


In [ ]:
# Compute bounds from your points
min_lat, max_lat = dfp["StartLat"].min(), dfp["StartLat"].max()
min_lon, max_lon = dfp["StartLon"].min(), dfp["StartLon"].max()

# Center map on mean of points
center_lat = dfp["StartLat"].mean()
center_lon = dfp["StartLon"].mean()

m = folium.Map(location=[center_lat, center_lon], tiles="CartoDB positron", zoom_start=5)

# Fit map to bounds (this is the "zoom" you want)
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

# Color palette (Folium has a fixed set; we cycle)
palette = [
    "red","blue","green","purple","orange","darkred","lightred","beige",
    "darkblue","darkgreen","cadetblue","darkpurple","white","pink",
    "lightblue","lightgreen","gray","black","lightgray"
]

folders = sorted(dfp["Folder"].astype(str).unique())
folder_to_color = {f: palette[i % len(palette)] for i, f in enumerate(folders)}

# Use clustering to keep map fast if many points
cluster = MarkerCluster(name="Start positions (clustered)").add_to(m)

# Add markers
for _, r in dfp.iterrows():
    folder = str(r["Folder"])
    lat = float(r["StartLat"])
    lon = float(r["StartLon"])

    tooltip = f"Folder: {folder}"
    popup = folium.Popup(
        html=f"""
        <b>Folder:</b> {folder}<br>
        <b>Wagon:</b> {r.get('Wagon', '')}<br>
        <b>Filename:</b> {r.get('Filename', '')}<br>
        <b>Date:</b> {r.get('Date', '')}<br>
        <b>StartTime:</b> {r.get('StartTime', '')}<br>
        """,
        max_width=450
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color=folder_to_color[folder],
        fill=True,
        fill_color=folder_to_color[folder],
        fill_opacity=0.85,
        tooltip=tooltip,
        popup=popup
    ).add_to(cluster)

# Add a simple legend (HTML overlay)
legend_items = "".join(
    [f"<li><span style='background:{folder_to_color[f]};'></span>{f}</li>" for f in folders[:25]]
)
legend_note = "" if len(folders) <= 25 else f"<div>Showing first 25 of {len(folders)} folders</div>"

legend_html = f"""
<div style="
position: fixed; 
bottom: 25px; left: 25px; width: 220px; z-index:9999;
background: white; padding: 10px; border: 1px solid #ccc;
font-size: 12px; border-radius: 6px;
">
<b>Folder (Monitoring kit)</b>
{legend_note}
<ul style="list-style:none; padding-left:0; margin: 8px 0 0 0;">
{legend_items}
</ul>
<style>
ul li {{margin: 2px 0;}}
ul li span {{
display:inline-block; width: 12px; height: 12px; margin-right: 6px;
vertical-align: middle; border: 1px solid #777;
}}
</style>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=True).add_to(m)


m.save("start_positions_by_folder.html")

m